In [ ]:
import cv2
from rapidocr_onnxruntime import RapidOCR

def run_ocr(image_path):
    # 1. RapidOCR 엔진 초기화 (사용자 지정 모델 경로 설정)
    # 주의: 아래 파일들이 프로젝트 폴더 내에 실제로 존재해야 합니다.
    engine = RapidOCR(
        det_model_path='ch_PP-OCRv4_det_infer.onnx',    # 텍스트 위치 감지 모델
        rec_model_path='model.onnx', # [중요] 한국어 텍스트 인식 모델
        rec_keys_path='korean_dict.txt'                # [중요] 한국어 사전 파일
    )

    print(f"[-] 이미지 처리 중: {image_path}")

    # 2. 이미지 읽기 및 추론 실행
    # result: 감지된 텍스트 정보 리스트, elapse: 소요 시간
    result, elapse = engine(image_path)

    if not result:
        print("[!] 텍스트를 찾을 수 없습니다.")
        return

    # 3. 결과 출력
    print(f"[-] 처리 시간: {elapse}초")
    print("-" * 30)

    # result 포맷: [[박스 좌표], '텍스트', 신뢰도]
    for i, item in enumerate(result):
        box, text, confidence = item
        print(f"[{i+1}] 텍스트: {text} (신뢰도: {confidence:.4f})")
        # print(f"    좌표: {box}") # 좌표가 필요하면 주석 해제

    print("-" * 30)

if __name__ == "__main__":
    # 테스트할 이미지 파일 경로 (jpg, png 등)
    target_image = 'unnamed.jpeg'

    # 이미지 파일 존재 여부 확인 (선택 사항)
    import os
    if os.path.exists(target_image):
        run_ocr(target_image)
    else:
        print(f"[Error] '{target_image}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

In [ ]:
import cv2
import time
from paddleocr import PaddleOCR

def fast_cpu_ocr(img_path):
    # ------------------------------------------------------------------
    # 1. 모델 초기화 (CPU 가속 설정)
    # ------------------------------------------------------------------
    ocr = PaddleOCR(
        lang='korean',
        use_gpu=False,           # GPU 끔
        enable_mkldnn=True,      # [핵심] CPU 연산 가속 활성화
        use_angle_cls=False,     # [핵심] 각도 분류 끔 (문서가 정방향이면 끄는 게 훨씬 빠름)
        ocr_version='PP-OCRv4',  # 최신 경량 모델 사용
        show_log=False           # 불필요한 로그 출력 방지
    )

    # ------------------------------------------------------------------
    # 2. 이미지 전처리 (리사이징)
    # ------------------------------------------------------------------
    start_time = time.time()

    img = cv2.imread(img_path)
    if img is None:
        print("이미지를 읽을 수 없습니다.")
        return

    # 원본 이미지 크기 확인
    h, w, _ = img.shape

    # [핵심] 이미지의 긴 변을 제한 (예: 960px ~ 1280px)
    # 너무 크면 CPU가 힘들어하고, 너무 작으면 인식이 안 됩니다.
    # 속도가 최우선이라면 960, 정확도와 타협하면 1280 추천
    max_side_limit = 960

    if max(h, w) > max_side_limit:
        scale = max_side_limit / max(h, w)
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

    # ------------------------------------------------------------------
    # 3. 인식 수행
    # ------------------------------------------------------------------
    # cls=False를 줘서 추론 단계에서도 방향 탐지를 확실히 배제
    result = ocr.ocr(img, cls=False)

    end_time = time.time()
    elapsed = end_time - start_time
    print(f"🚀 처리 완료: {elapsed:.4f}초 소요 (해상도: {img.shape[1]}x{img.shape[0]})")

    # 결과 출력
    if result and result[0]:
        for line in result[0]:
            print(f"텍스트: {line[1][0]}")
    else:
        print("검출된 텍스트가 없습니다.")

# 실행 예시
if __name__ == '__main__':
    fast_cpu_ocr('your_image.jpg')